In [7]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

words = open('names.txt', 'r').read().splitlines()

In [8]:
# So the paper is predicting the next word - we will be predicting the next character
# In paper, model embedds 17,000 words into 30 dimensional vector space which encodes he meaning of words
# model tries to simultaneously learn the word feature vectors and the parameters of that probability function
# So I think this means learn the position of the word vector in that space
# as well as learn the actual space dimensions??? so this reminds me of PCA: what dimensions carry the most variability / information about the word's meaning?

In [9]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [10]:
# built the training dataset

block_size = 3 # context length: how many previous chars do we use to predict the next?
X, Y = [], [] # X are the nn inputs, Y are the labels for each example in X
for w in words:
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix] # crop and append (so this just shifts the context down by one character so that we eventually get all 3 consecutive char groups for each word, starting with ...)

X = torch.tensor(X)
Y = torch.tensor(Y)

In [20]:
print(X, Y)
print(X.shape, X.dtype, Y.shape, Y.dtype)

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        ...,
        [26, 26, 25],
        [26, 25, 26],
        [25, 26, 24]]) tensor([ 5, 13, 13,  ..., 26, 24,  0])
torch.Size([228146, 3]) torch.int64 torch.Size([228146]) torch.int64


In [45]:
# Construct Embedding: the look-up table C
# this the look-up table is how we get the parameters of each char (the parameters representing the semantic meaning of words in the paper)
# We will embed the 27 chars into a 2D space
# or are we embedding all examples (X.shape[0] = 228146) into a 2D space?

C = torch.randn((27, 2))
emb = C[X]
print(emb.shape)
print(C.shape)
# So I believe this means that we have 228146 context examples in X, each of which is list of 3 numbers
# and we have now embedded each into a 2D vector space
print(X[13, 2]) # this is the 2nd number (index of char) in the 13th context input example
print(C[X][13, 2]) # this is the 2D embedding vector of that ^ char
# and we see..
print(C[1])

torch.Size([228146, 3, 2])
torch.Size([27, 2])
tensor(1)
tensor([-1.4391,  1.2845])
tensor([-1.4391,  1.2845])


In [47]:
# Construct Hidden Activation Layer (the nonlinearity, tanh)
# Number of inputs to this layer is 3*2 (from emb because we have 3 previous characters, each as a 2D vector)
# Number of neurons in this layer is a hyperparameter (up to us, can be optimized)

W1 = torch.randn((6, 100)) # weights
b1 = torch.randn(100) # biases

# vid 3, around 26 minutes
# we want to concatenate the three previous context char embeddings into a single 6D vector so that we can perform a forward pass
# torch.cat(torch.unbind(emb, 1), 1) # THESE LINES GIVE THE SAME RESULT
# emb.view(X.shape[0], 6) # THESE LINES GIVE THE SAME RESULT but this one is cheaper due to PyTorch's memory managament

# Thus, hidden layer:
h = torch.tanh(emb.view(emb.shape[0], 6) @ W1 + b1)

In [55]:
# Final Layer: (Softmax) Output Layer
W2 = torch.randn((100, 27)) # weights of layer
b2 = torch.randn(27) # biases of layer

logits = h @ W2 + b2 # forward pass into this layer
counts = logits.exp() # apply softmax activation onto layer output
prob = counts / counts.sum(1, keepdim=True) # apply softmax activation onto layer output
prob.shape

torch.Size([228146, 27])

In [59]:
# Loss (NLL of the Training Data as Predicted by the NN)
# prob[torch.arange(X.shape[0]), Y] is indexing into prob matrix (tensor) and getting the prob that the model assigned to the actual output Y for every example
loss = -prob[torch.arange(X.shape[0]), Y].log().mean()

In [64]:
# =============== WRITTEN NICELY and improved =======================
X.shape, Y.shape

(torch.Size([228146, 3]), torch.Size([228146]))

In [65]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [66]:
sum(p.nelement() for p in parameters) # total number of parameters in model

3481

In [72]:
emb = C[X]
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
logits = h @ W2 + b2 # log-counts
loss = F.cross_entropy(logits, Y)